In [ ]:
from pathlib import Path
import sys

# Ensure the repository root is importable when running from the project root
# or from the analysis directory.
REPO_ROOT = Path.cwd().resolve()
if REPO_ROOT.name == 'analysis':
    REPO_ROOT = REPO_ROOT.parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))
print("Added REPO_ROOT to PYTHONPATH:", REPO_ROOT)


# Qualitative Case Selection Workbook

- Purpose: build clinician-review cases where all models are compared on the same `eval_seed` / initial scenario.
- Outputs: `selected_cases.csv`, `case_sequences.csv`, and `review_payload.json`.
- Each selected case contains one actual trajectory per model, not a modal trajectory pooled across different episodes.
- To rerun on new data, update the configuration section below.

## 1. Load Configuration and File Paths

Edit these values to rerun on new trajectories or a different manifest.
If your trajectories live under `results/<model>/<run>/trajectories.csv`, set `use_results_root = True`.

What the notebook actually needs:
- one `trajectories.csv` per model
- required columns: `eval_seed`, `t`, `state_id`, `sofa_score`, `action_idx`, `reward`, `done`, `survived`
- either `manifest.json` plus the run folders, or direct `csv_paths`
- `dynamics.npz` for state-cluster summaries and policy labels

What your friends will usually change:
- `dataset_tag`
- `methods`
- `results_root` and `run_selector` if they use the `results/<model>/<run>/trajectories.csv` layout
- `csv_paths` if they want to bypass the manifest entirely
- `manifest_path` only if they are using the manifest-based loader
- `dynamics_path` if the benchmark assets live somewhere else

In [ ]:
from pathlib import Path
import json
import sys
import pandas as pd
import numpy as np

REPO_ROOT = Path.cwd().resolve()
if REPO_ROOT.name == 'analysis':
    REPO_ROOT = REPO_ROOT.parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from analysis import qualitative_cases as qc


def resolve_path(path_str: str) -> Path:
    path = Path(path_str)
    return path if path.is_absolute() else (REPO_ROOT / path)


def ensure_manifest(path: Path) -> None:
    if not path.exists():
        raise FileNotFoundError(f"Manifest not found: {path}")


In [ ]:
# For the common case, the notebook reads trajectories directly from `results_root`.
# `manifest_path` is only used if `use_results_root = False`.
manifest_path = resolve_path(
    "runs/offline-dev-expert-5k-50k-100k-v5/manifest.json"
)
dataset_tag = "5k"
methods = [
    "dqn_offline",
    "ppo_logged_offline",
    "raw_model_discrete_cem_offline",
    "reconstructive_model_continuous_cem_offline",
]

horizon = 3
# For matched eval_seed cases, support is one actual rollout per model.
# Retained for compatibility with earlier exports.
min_support = 1
length_bucket_size = 8
top_k = 8
stratify_by_outcome = False
max_episode_length = None  # set to 8 to force <= 8 observed rows per model

# Seed-matched clinician review defaults.
selection_unit = "eval_seed_initial_state"
require_same_initial_state = True
require_same_length_bucket = True
include_identical_model_trajectories = False

# Clinician-review case mix: longer, divergent, and outcome-balanced.
# The selector first tries `preferred_min_median_length`; if a required group has no
# candidates, it falls back to `fallback_min_median_length`.
preferred_min_median_length = 6
fallback_min_median_length = 4
review_max_episode_length = 10
minimum_divergent_score = 0.40
selection_group_targets = {
    "all_died": 1,
    "all_survived": 1,
    "mixed_outcome": top_k - 2,
}

output_dir = resolve_path("reports/qual_cases_5k_ms8_4models")
dynamics_path = qc.DEFAULT_DYNAMICS_PATH

use_results_root = True
results_root = resolve_path("results")
run_selector = "latest"

# Optional: bypass the manifest and provide direct CSV paths per method.
# csv_paths = {
#     "dqn_offline": resolve_path("path/to/dqn/trajectories.csv"),
#     "ppo_logged_offline": resolve_path("path/to/ppo/trajectories.csv"),
#     "raw_model_discrete_cem_offline": resolve_path("path/to/raw_model/trajectories.csv"),
#     "reconstructive_model_continuous_cem_offline": resolve_path("path/to/reconstructive_model/trajectories.csv"),
# }
csv_paths = None

## 2. Load and Validate Trajectory CSVs

This section loads the trajectories for the selected methods and checks required columns.

In [ ]:
required_cols = {
    "eval_seed",
    "t",
    "state_id",
    "sofa_score",
    "action_idx",
    "reward",
    "done",
    "survived",
}


def load_manifest_csv_paths() -> dict[str, Path]:
    ensure_manifest(manifest_path)
    rows = json.loads(manifest_path.read_text(encoding="utf-8"))
    by_method = {}
    for row in rows:
        if row.get("dataset_tag") != dataset_tag:
            continue
        method_id = row.get("method_id")
        if method_id in methods:
            by_method[method_id] = resolve_path(row["output_dir"]) / "trajectories.csv"
    missing = [method for method in methods if method not in by_method]
    if missing:
        raise ValueError(f"Missing methods in manifest: {missing}")
    return by_method


def find_run_dir(method_root: Path, selector: str) -> Path:
    if selector not in {"latest", "first"}:
        candidate = method_root / selector
        if candidate.exists():
            return candidate
        raise FileNotFoundError(f"Run folder not found: {candidate}")
    run_dirs = [path for path in method_root.iterdir() if path.is_dir()]
    if not run_dirs:
        raise FileNotFoundError(f"No run folders under: {method_root}")
    if selector == "first":
        return sorted(run_dirs)[0]
    return max(run_dirs, key=lambda path: path.stat().st_mtime)


def load_results_csv_paths() -> dict[str, Path]:
    by_method = {}
    for method_id in methods:
        method_root = results_root / method_id
        run_dir = find_run_dir(method_root, run_selector)
        path = run_dir / "trajectories.csv"
        if not path.exists():
            raise FileNotFoundError(f"Missing trajectories: {path}")
        by_method[method_id] = path
    return by_method


def load_trajectories(path: Path) -> pd.DataFrame:
    df = pd.read_csv(path)
    missing = required_cols.difference(df.columns)
    if missing:
        raise ValueError(f"Missing columns in {path}: {sorted(missing)}")
    return df


if csv_paths is None:
    csv_paths = load_results_csv_paths() if use_results_root else load_manifest_csv_paths()

trajectories_by_method: dict[str, pd.DataFrame] = {}
for method_id, path in csv_paths.items():
    df = load_trajectories(path)
    trajectories_by_method[method_id] = df
    lengths = df.groupby("eval_seed").size()
    print(
        f"{method_id}: episodes={lengths.shape[0]} rows={len(df)} "
        f"len[min,median,max]=({lengths.min()}, {int(lengths.median())}, {lengths.max()})"
    )

In [ ]:
# Clinician-facing trajectory review should compare the same eval_seed across models.
# Same eval_seed + same initial_state gives the closest available "same patient/scenario" comparison.
strict_same_eval_seed = True

sets = {m: set(df["eval_seed"].unique()) for m, df in trajectories_by_method.items()}
common_eval_seeds = set.intersection(*sets.values()) if sets else set()
if not common_eval_seeds:
    raise ValueError("No eval_seed values are shared by all selected methods.")

print("Common eval_seed count:", len(common_eval_seeds))
for m, s in sets.items():
    print(f"{m}: episodes={len(s)} (kept={len(common_eval_seeds)})")

for m in list(trajectories_by_method.keys()):
    trajectories_by_method[m] = trajectories_by_method[m][
        trajectories_by_method[m]["eval_seed"].isin(common_eval_seeds)
    ].copy()

for method_id, df in trajectories_by_method.items():
    lengths = df.groupby("eval_seed").size()
    print(f"After eval_seed filter - {method_id}: episodes={lengths.shape[0]} rows={len(df)}")

## 3. Normalize and Filter Trajectory Data

The data are already normalized in the benchmark. This step focuses on filtering invalid rows and optionally restricting episode length.

In [ ]:
filtered_by_method: dict[str, pd.DataFrame] = {}

for method_id, df in trajectories_by_method.items():
    clean = df.dropna(subset=required_cols).copy()
    clean = clean.sort_values(["eval_seed", "t"]).reset_index(drop=True)
    if max_episode_length is not None:
        lengths = clean.groupby("eval_seed").size()
        keep = lengths[lengths <= max_episode_length].index
        clean = clean[clean["eval_seed"].isin(keep)].copy()
    filtered_by_method[method_id] = clean

    lengths = clean.groupby("eval_seed").size()
    print(
        f"{method_id}: episodes={lengths.shape[0]} rows={len(clean)} "
        f"len[min,median,max]=({lengths.min()}, {int(lengths.median())}, {lengths.max()})"
    )

## 4. Qualitative Case Selection Rules

Rules applied below:
- The selection unit is `eval_seed + initial_state`, so the doctor reviews the same starting scenario across models.
- All selected models must contain the same `eval_seed`.
- By default, all models must have the same `initial_state` and same `length_bucket` for that seed.
- Each per-model row is the actual full trajectory for that model/seed, capped only by `review_max_episode_length` during selection.
- The final outcome (`survived`) and observed episode length remain model-specific and are exported.
- Selection prioritizes divergent cases that are long enough to be meaningful but short enough for full review, capped by `review_max_episode_length`.
- The final mix targets at least one unanimous death case, one unanimous survival case, and fills the rest with mixed-outcome divergent cases.

In [ ]:
dynamics = qc._load_dynamics(dynamics_path)
centers = np.asarray(dynamics["state_cluster_centers"], dtype=np.float64)
sofa_scores = np.asarray(dynamics["sofa_scores"], dtype=np.float64)
expert_policy = np.asarray(dynamics["expert_policy"], dtype=np.float64)
optimal_policy = qc._compute_optimal_policy(
    np.asarray(dynamics["tx_mat"], dtype=np.float64),
    np.asarray(dynamics["r_mat"], dtype=np.float64),
    gamma=1.0,
)

def build_full_episode_sequences(df: pd.DataFrame) -> list[qc.EpisodeSequence]:
    """Build one full, variable-length sequence per eval_seed.

    Unlike qc.build_episode_sequences(..., horizon=...), this keeps every action in
    the episode so clinician-review exports can cover the full trajectory.
    """
    sequences: list[qc.EpisodeSequence] = []
    for eval_seed, episode in df.groupby("eval_seed", sort=False):
        episode = episode.sort_values("t").reset_index(drop=True)
        if episode.empty:
            continue
        action_seq = tuple(int(value) for value in episode["action_idx"].tolist())
        state_seq: list[int] = [int(episode.loc[0, "state_id"])]
        for step in range(len(episode)):
            if step + 1 < len(episode):
                next_state = int(episode.loc[step + 1, "state_id"])
            else:
                next_state = qc._terminal_state_id(episode.loc[step])
            state_seq.append(next_state)
        sequences.append(
            qc.EpisodeSequence(
                eval_seed=int(eval_seed),
                initial_state=int(episode.loc[0, "state_id"]),
                initial_sofa=float(episode.loc[0, "sofa_score"]),
                survived=bool(episode.loc[len(episode) - 1, "survived"]),
                action_seq=action_seq,
                state_seq=tuple(state_seq),
                length=int(len(episode)),
            )
        )
    return sequences


sequences_by_method: dict[str, list[qc.EpisodeSequence]] = {}
for method_id, df in filtered_by_method.items():
    sequences_by_method[method_id] = build_full_episode_sequences(df)


def sequence_frame(method_id: str, seqs: list[qc.EpisodeSequence]) -> pd.DataFrame:
    rows = []
    for seq in seqs:
        length_bucket = qc._length_bucket(seq.length, length_bucket_size)
        rows.append(
            {
                "method_id": method_id,
                "method_label": qc.DISPLAY_NAMES.get(method_id, method_id),
                "eval_seed": seq.eval_seed,
                "initial_state": seq.initial_state,
                "initial_sofa": seq.initial_sofa,
                "survived": seq.survived,
                "outcome_label": "survived" if seq.survived else "died",
                "length": seq.length,
                "length_bucket": length_bucket,
                "length_bucket_min": qc._length_bucket_bounds(length_bucket, length_bucket_size)[0],
                "length_bucket_max": qc._length_bucket_bounds(length_bucket, length_bucket_size)[1],
                "action_seq": seq.action_seq,
                "state_seq": seq.state_seq,
            }
        )
    return pd.DataFrame(rows)


frames_by_method = {
    method_id: sequence_frame(method_id, seqs)
    for method_id, seqs in sequences_by_method.items()
}

common_eval_seeds = set.intersection(
    *[set(frame["eval_seed"].unique()) for frame in frames_by_method.values()]
)

candidate_rows: list[dict] = []
sequence_rows: list[dict] = []
for eval_seed in sorted(common_eval_seeds):
    rows = []
    for method_id in methods:
        frame = frames_by_method[method_id]
        match = frame[frame["eval_seed"] == eval_seed]
        if match.empty:
            rows = []
            break
        rows.append(match.iloc[0].to_dict())
    if len(rows) != len(methods):
        continue

    initial_states = {int(row["initial_state"]) for row in rows}
    if require_same_initial_state and len(initial_states) != 1:
        continue
    length_buckets = {int(row["length_bucket"]) for row in rows}
    if require_same_length_bucket and len(length_buckets) != 1:
        continue
    if stratify_by_outcome and len({bool(row["survived"]) for row in rows}) != 1:
        continue

    initial_state = int(rows[0]["initial_state"])
    length_bucket = int(rows[0]["length_bucket"])
    candidate_key = f"seed{int(eval_seed)}:state{initial_state}:len{length_bucket}"

    action_sequences = [tuple(row["action_seq"]) for row in rows]
    state_sequences = [tuple(row["state_seq"]) for row in rows]
    survived_values = [bool(row["survived"]) for row in rows]

    def pairwise_sequence_distance(values: list[tuple[int, ...]]) -> float:
        distances = []
        for idx, first in enumerate(values):
            for second in values[idx + 1:]:
                limit = min(len(first), len(second))
                mismatches = sum(1 for pos in range(limit) if first[pos] != second[pos])
                mismatches += abs(len(first) - len(second))
                distances.append(mismatches / max(len(first), len(second), 1))
        return float(np.mean(distances)) if distances else 0.0

    action_disagreement = pairwise_sequence_distance(action_sequences)
    state_disagreement = pairwise_sequence_distance(state_sequences)
    outcome_disagreement = float(len(set(survived_values)) > 1)
    disagreement_score = float(0.65 * action_disagreement + 0.25 * state_disagreement + 0.10 * outcome_disagreement)
    unique_joint_sequences = len(set(zip(action_sequences, state_sequences)))
    if unique_joint_sequences < 2 and not include_identical_model_trajectories:
        continue

    lengths = [int(row["length"]) for row in rows]
    candidate_rows.append(
        {
            "candidate_key": candidate_key,
            "eval_seed": int(eval_seed),
            "initial_state": initial_state,
            "length_bucket": length_bucket,
            "length_bucket_min": qc._length_bucket_bounds(length_bucket, length_bucket_size)[0],
            "length_bucket_max": qc._length_bucket_bounds(length_bucket, length_bucket_size)[1],
            "survived": None if not stratify_by_outcome else survived_values[0],
            "initial_sofa": float(sofa_scores[initial_state]),
            "coverage_raw": len(rows),
            "coverage_mean": float(len(rows)),
            "episode_length_min": min(lengths),
            "episode_length_median": float(np.median(lengths)),
            "episode_length_max": max(lengths),
            "outcome_values": "|".join("survived" if value else "died" for value in survived_values),
            "outcome_disagreement": outcome_disagreement,
            "action_disagreement": action_disagreement,
            "state_disagreement": state_disagreement,
            "disagreement_score": disagreement_score,
            "stability_score": 1.0,
            "unique_modal_joint_sequences": unique_joint_sequences,
            "valid_comparison": True,
            "state_summary": qc._state_summary_text(initial_state, centers),
            "expert_top_action": qc._expert_top_action(initial_state, expert_policy)[0],
            "expert_top_action_label": qc._action_label(qc._expert_top_action(initial_state, expert_policy)[0]),
            "expert_top_action_prob": qc._expert_top_action(initial_state, expert_policy)[1],
            "selection_score": disagreement_score,
        }
    )

    for row in rows:
        action_seq = tuple(row["action_seq"])
        state_seq = tuple(row["state_seq"])
        step_rows = qc._step_rows_from_joint_mode(action_seq, state_seq, sofa_scores, optimal_policy)
        sequence_rows.append(
            {
                "candidate_key": candidate_key,
                "eval_seed": int(eval_seed),
                "initial_state": initial_state,
                "length_bucket": length_bucket,
                "length_bucket_min": qc._length_bucket_bounds(length_bucket, length_bucket_size)[0],
                "length_bucket_max": qc._length_bucket_bounds(length_bucket, length_bucket_size)[1],
                "survived": bool(row["survived"]),
                "outcome_label": row["outcome_label"],
                "episode_length": int(row["length"]),
                "method_id": row["method_id"],
                "method_label": row["method_label"],
                "support": 1,
                "modal_step0_action": int(action_seq[0]) if action_seq else -1,
                "modal_step0_action_label": qc._action_label(int(action_seq[0])) if action_seq else "",
                "modal_action_seq_ids": "|".join(str(value) for value in action_seq),
                "modal_action_seq_labels": qc._format_action_sequence(action_seq),
                "modal_action_seq_prob": 1.0,
                "modal_state_seq_ids": "|".join(str(value) for value in state_seq),
                "modal_state_seq_labels": qc._format_state_sequence_ids(state_seq),
                "modal_state_seq_sofa": qc._format_state_sequence_sofa(state_seq, sofa_scores),
                "modal_state_seq_prob": 1.0,
                "modal_joint_prob": 1.0,
                "modal_joint_count": 1,
                "modal_joint_action_seq_ids": "|".join(str(value) for value in action_seq),
                "modal_joint_action_seq_labels": qc._format_action_sequence(action_seq),
                "modal_joint_state_seq_ids": "|".join(str(value) for value in state_seq),
                "modal_joint_state_seq_labels": qc._format_state_sequence_ids(state_seq),
                "modal_joint_state_seq_sofa": qc._format_state_sequence_sofa(state_seq, sofa_scores),
                "modal_joint_steps_json": json.dumps(step_rows, ensure_ascii=True),
                "stability_score": 1.0,
                "trajectory_kind": "actual_seed_matched",
            }
        )

candidates = pd.DataFrame(candidate_rows)
sequences = pd.DataFrame(sequence_rows)
if candidates.empty:
    raise ValueError(
        "No seed-matched candidates left after filtering. Try setting "
        "require_same_length_bucket=False or lowering horizon/max_episode_length constraints."
    )

def outcome_group_from_values(outcome_values: str) -> str:
    values = outcome_values.split("|")
    if all(value == "died" for value in values):
        return "all_died"
    if all(value == "survived" for value in values):
        return "all_survived"
    return "mixed_outcome"


candidates["outcome_group"] = candidates["outcome_values"].map(outcome_group_from_values)


def select_ranked_group(
    pool: pd.DataFrame,
    group_name: str,
    n: int,
    used_initial_states: set[int],
    used_candidate_keys: set[str],
) -> pd.DataFrame:
    if n <= 0:
        return pool.head(0).copy()
    group = pool[
        (pool["outcome_group"] == group_name)
        & (~pool["candidate_key"].isin(used_candidate_keys))
        & (~pool["initial_state"].isin(used_initial_states))
        & (pool["episode_length_max"] <= review_max_episode_length)
    ].copy()
    if group.empty:
        return group

    preferred = group[
        (group["episode_length_median"] >= preferred_min_median_length)
        & (group["disagreement_score"] >= minimum_divergent_score)
    ].copy()
    if preferred.empty:
        preferred = group[
            (group["episode_length_median"] >= fallback_min_median_length)
            & (group["disagreement_score"] >= minimum_divergent_score)
        ].copy()
    if preferred.empty:
        preferred = group[group["episode_length_median"] >= preferred_min_median_length].copy()
    if preferred.empty:
        preferred = group[group["episode_length_median"] >= fallback_min_median_length].copy()
    if preferred.empty:
        preferred = group.copy()

    preferred = preferred.sort_values(
        ["disagreement_score", "episode_length_median", "episode_length_min"],
        ascending=[False, False, False],
    )
    return preferred.head(n).copy()


selected_parts = []
used_initial_states: set[int] = set()
used_candidate_keys: set[str] = set()
for group_name, group_n in selection_group_targets.items():
    part = select_ranked_group(
        candidates,
        group_name=group_name,
        n=group_n,
        used_initial_states=used_initial_states,
        used_candidate_keys=used_candidate_keys,
    )
    if part.empty:
        print(f"Warning: no candidates found for requested group {group_name!r}")
        continue
    part["selection_bucket"] = group_name
    part["bucket_rank"] = np.arange(1, len(part) + 1)
    selected_parts.append(part)
    used_initial_states.update(int(value) for value in part["initial_state"].tolist())
    used_candidate_keys.update(part["candidate_key"].tolist())

selected = pd.concat(selected_parts, ignore_index=True) if selected_parts else candidates.head(0).copy()
if len(selected) < top_k:
    filler = candidates[
        (~candidates["candidate_key"].isin(set(selected["candidate_key"])))
        & (~candidates["initial_state"].isin(set(selected["initial_state"])))
    ].copy()
    filler = filler[
        (filler["episode_length_median"] >= fallback_min_median_length)
        & (filler["episode_length_max"] <= review_max_episode_length)
    ]
    filler = filler.sort_values(
        ["outcome_disagreement", "disagreement_score", "episode_length_median"],
        ascending=[False, False, False],
    ).head(top_k - len(selected))
    filler["selection_bucket"] = "filler_divergent"
    filler["bucket_rank"] = np.arange(1, len(filler) + 1)
    selected = pd.concat([selected, filler], ignore_index=True)

selected = selected.sort_values(
    ["selection_bucket", "bucket_rank"],
    ascending=[True, True],
).reset_index(drop=True)
selected["case_rank"] = np.arange(1, len(selected) + 1)

selected_case_keys = set(selected["candidate_key"].tolist())
selected_sequences = sequences[sequences["candidate_key"].isin(selected_case_keys)].copy()
case_rank_map = dict(zip(selected["candidate_key"], selected["case_rank"]))
selected_sequences["case_rank"] = selected_sequences["candidate_key"].map(case_rank_map)
selected_sequences = selected_sequences.sort_values(["case_rank", "method_label"]).reset_index(drop=True)

print(f"Seed-matched candidates: {len(candidates)}")
print(f"Selected cases: {len(selected)}")
print(f"Selected trajectories: {len(selected_sequences)}")
print(
    selected[
        [
            "case_rank",
            "candidate_key",
            "selection_bucket",
            "outcome_group",
            "eval_seed",
            "initial_state",
            "episode_length_min",
            "episode_length_median",
            "episode_length_max",
            "outcome_values",
            "disagreement_score",
        ]
    ]
)

## 5. Feature Engineering for Four Models

Build compact feature tables per model to support model-level inspection and comparison.

In [ ]:
features_by_method: dict[str, pd.DataFrame] = {}
for method_id, seqs in sequences_by_method.items():
    features_by_method[method_id] = pd.DataFrame(
        {
            "eval_seed": [seq.eval_seed for seq in seqs],
            "initial_state": [seq.initial_state for seq in seqs],
            "length": [seq.length for seq in seqs],
            "length_bucket": [qc._length_bucket(seq.length, length_bucket_size) for seq in seqs],
            "action_seq": [seq.action_seq for seq in seqs],
            "state_seq": [seq.state_seq for seq in seqs],
            "survived": [seq.survived for seq in seqs],
        }
    )

for method_id, feats in features_by_method.items():
    print(method_id, feats.shape)

## 6. Train/Run Model 1 (DQN)

Summary for DQN trajectories used in the selection.

In [ ]:
method_id = "dqn_offline"
feats = features_by_method[method_id]

print(feats.head())
print("Top initial_state:")
print(feats["initial_state"].value_counts().head(5))
print("Top action_seq:")
print(feats["action_seq"].value_counts().head(5))

## 7. Train/Run Model 2 (PPO)

Summary for PPO trajectories used in the selection.

In [ ]:
method_id = "ppo_logged_offline"
feats = features_by_method[method_id]

print(feats.head())
print("Top initial_state:")
print(feats["initial_state"].value_counts().head(5))
print("Top action_seq:")
print(feats["action_seq"].value_counts().head(5))

## 8. Train/Run Model 3 (Raw-D)

Summary for Raw-D trajectories used in the selection.

In [ ]:
method_id = "raw_model_discrete_cem_offline"
feats = features_by_method[method_id]

print(feats.head())
print("Top initial_state:")
print(feats["initial_state"].value_counts().head(5))
print("Top action_seq:")
print(feats["action_seq"].value_counts().head(5))

## 9. Train/Run Model 4 (Reconstructive-C)

Summary for Reconstructive-C trajectories used in the selection.

In [ ]:
method_id = "reconstructive_model_continuous_cem_offline"
feats = features_by_method[method_id]

print(feats.head())
print("Top initial_state:")
print(feats["initial_state"].value_counts().head(5))
print("Top action_seq:")
print(feats["action_seq"].value_counts().head(5))

## 10. Compare Model Outputs Across Trajectories

Aggregate per-case outputs to compare modal action and state sequences across models.

In [ ]:
comparison_actions = selected_sequences.pivot_table(
    index="candidate_key",
    columns="method_label",
    values="modal_action_seq_labels",
    aggfunc="first",
)
comparison_states = selected_sequences.pivot_table(
    index="candidate_key",
    columns="method_label",
    values="modal_state_seq_labels",
    aggfunc="first",
)
comparison_outcomes = selected_sequences.pivot_table(
    index="candidate_key",
    columns="method_label",
    values="outcome_label",
    aggfunc="first",
)

print("Actual seed-matched actions by case:")
print(comparison_actions)
print("\nActual seed-matched states by case:")
print(comparison_states)
print("\nOutcomes by case:")
print(comparison_outcomes)

## 11. Export Results and Artifacts

Write CSV/JSON outputs for downstream review and sharing.

In [ ]:
output_dir.mkdir(parents=True, exist_ok=True)

candidates.to_csv(output_dir / "initial_state_candidates.csv", index=False)
selected.to_csv(output_dir / "selected_cases.csv", index=False)
selected_sequences.to_csv(output_dir / "case_sequences.csv", index=False)


def build_seed_matched_review_payload(selected: pd.DataFrame, selected_sequences: pd.DataFrame) -> dict[str, object]:
    cases = []
    for _, case in selected.iterrows():
        case_rows = selected_sequences[selected_sequences["candidate_key"] == case["candidate_key"]].copy()
        method_rows = []
        for _, row in case_rows.iterrows():
            method_rows.append(
                {
                    "method_id": row["method_id"],
                    "method_label": row["method_label"],
                    "eval_seed": int(row["eval_seed"]),
                    "episode_length": int(row["episode_length"]),
                    "survived": bool(row["survived"]),
                    "outcome_label": row["outcome_label"],
                    "trajectory_kind": row["trajectory_kind"],
                    "action_seq_ids": row["modal_joint_action_seq_ids"],
                    "action_seq_labels": row["modal_joint_action_seq_labels"],
                    "state_seq_ids": row["modal_joint_state_seq_ids"],
                    "state_seq_labels": row["modal_joint_state_seq_labels"],
                    "state_seq_sofa": row["modal_joint_state_seq_sofa"],
                    "steps": json.loads(row["modal_joint_steps_json"]),
                }
            )
        cases.append(
            {
                "case_rank": int(case["case_rank"]),
                "candidate_key": case["candidate_key"],
                "selection_unit": selection_unit,
                "eval_seed": int(case["eval_seed"]),
                "initial_state": int(case["initial_state"]),
                "length_bucket": int(case["length_bucket"]),
                "length_bucket_min": int(case["length_bucket_min"]),
                "length_bucket_max": int(case["length_bucket_max"]),
                "episode_length_min": int(case["episode_length_min"]),
                "episode_length_median": float(case["episode_length_median"]),
                "episode_length_max": int(case["episode_length_max"]),
                "outcome_values": case["outcome_values"],
                "outcome_disagreement": float(case["outcome_disagreement"]),
                "action_disagreement": float(case["action_disagreement"]),
                "state_disagreement": float(case["state_disagreement"]),
                "disagreement_score": float(case["disagreement_score"]),
                "selection_bucket": case["selection_bucket"],
                "state_summary": case["state_summary"],
                "methods": method_rows,
            }
        )
    return {
        "version": 2,
        "selection_unit": selection_unit,
        "methods": methods,
        "cases": cases,
    }


payload = build_seed_matched_review_payload(selected, selected_sequences)
(output_dir / "review_payload.json").write_text(
    json.dumps(payload, indent=2, ensure_ascii=True),
    encoding="utf-8",
)

comparison_actions.to_csv(output_dir / "comparison_actions.csv")
comparison_states.to_csv(output_dir / "comparison_states.csv")
comparison_outcomes.to_csv(output_dir / "comparison_outcomes.csv")

metadata = {
    "manifest_path": str(manifest_path),
    "dataset_tag": dataset_tag,
    "methods": methods,
    "selection_unit": selection_unit,
    "strict_same_eval_seed": strict_same_eval_seed,
    "require_same_initial_state": require_same_initial_state,
    "require_same_length_bucket": require_same_length_bucket,
    "include_identical_model_trajectories": include_identical_model_trajectories,
    "preferred_min_median_length": preferred_min_median_length,
    "fallback_min_median_length": fallback_min_median_length,
    "review_max_episode_length": review_max_episode_length,
    "minimum_divergent_score": minimum_divergent_score,
    "selection_group_targets": selection_group_targets,
    "horizon": horizon,
    "step_export_mode": "full_selected_trajectory",
    "min_support": min_support,
    "length_bucket_size": length_bucket_size,
    "top_k": top_k,
    "stratify_by_outcome": stratify_by_outcome,
    "max_episode_length": max_episode_length,
    "dynamics_path": str(dynamics_path),
}
(output_dir / "selection_metadata.json").write_text(
    json.dumps(metadata, indent=2, ensure_ascii=True),
    encoding="utf-8",
)

print(f"Wrote seed-matched outputs to: {output_dir}")

## How to Read a Case

- `candidate_key` identifies the matched scenario: `seed<eval_seed>:state<initial_state>:len<length_bucket>`.
- Each row in `case_sequences.csv` is one actual model trajectory for the same `eval_seed`, not a modal trajectory from pooled episodes.
- `modal_*` columns are retained for compatibility with older exports, but now contain the actual truncated sequence for that model and seed.
- `outcome_label`, `survived`, and `episode_length` are model-specific and should be used when judging the whole trajectory.
- `modal_joint_steps_json` has the step-by-step state/action/next-state details for the review UI.
- Use `comparison_outcomes.csv` together with `comparison_actions.csv` and `comparison_states.csv` to sanity-check model differences.

## 12. Export Adapted State Interpretations for the Review UI

This optional export creates a compact state lookup table for the clinician-facing UI.
It keeps one row per state used in `review_payload.json` and removes the `p≈...` prefix from binary-feature interpretations, keeping only the semantic conclusion such as `valor 1 comum neste cluster`.


In [ ]:
def split_adapted_interpretation(text: str) -> tuple[str, str]:
    """Return semantic interpretation and optional estimated p-value text."""
    if "; " in text and text.startswith("p≈"):
        p_value, semantic = text.split("; ", 1)
        return semantic, p_value
    return text, ""


def clean_adapted_interpretation(text: str) -> str:
    return text.replace(" em relacao ao cohort usado para construir o MDP", "")


payload = json.loads((output_dir / "review_payload.json").read_text(encoding="utf-8"))
state_ids = set()
for case in payload["cases"]:
    state_ids.add(int(case["initial_state"]))
    for method in case["methods"]:
        for step in method["steps"]:
            for key in ["current_state", "next_state"]:
                value = step.get(key)
                if value is not None and int(value) not in qc.TERMINAL_STATE_LABELS:
                    state_ids.add(int(value))

adapted_rows = []
long_rows = []
for state_id in sorted(state_ids):
    center = centers[state_id]
    wide_row = {"state_id": state_id}
    for feature_name, value in zip(qc.FEATURE_NAMES, center):
        _, interpretation = qc._feature_interpretation(feature_name, float(value))
        adapted, p_estimated = split_adapted_interpretation(interpretation)
        adapted = clean_adapted_interpretation(adapted)

        # Wide export: one readable interpretation column per feature, plus an
        # optional p column for binary features where prevalence is estimable.
        wide_row[feature_name] = adapted
        wide_row[f"{feature_name}_p_estimated"] = p_estimated

        # Long export: easier for UI/database ingestion.
        long_rows.append(
            {
                "state_id": state_id,
                "feature_name": feature_name,
                "adapted_interpretation": adapted,
                "p_estimated": p_estimated,
            }
        )
    adapted_rows.append(wide_row)

adapted_wide_path = output_dir / "review_state_adapted_interpretations.csv"
adapted_long_path = output_dir / "review_state_adapted_interpretations_long.csv"
pd.DataFrame(adapted_rows).to_csv(adapted_wide_path, index=False)
pd.DataFrame(long_rows).to_csv(adapted_long_path, index=False)

print(f"Wrote adapted wide state interpretations to: {adapted_wide_path}")
print(f"Wrote adapted long state interpretations to: {adapted_long_path}")
print(f"States exported: {len(adapted_rows)}")